# People Aggregation Methodology

This notebook develops and evaluates the people aggregation method before the stable parts are moved into `pkb.models.people`.

The goal is to compare rule-based name/date matching against an identifier-derived validation dataset. Identifier matches are treated as a high-confidence silver standard rather than unquestionable truth.

## Setup

Run the source-specific build tasks first if any of these parquets are missing:

```bash
uv run luigi --module pkb.tasks.build.people.people PeopleTask --local-scheduler
```

In [ ]:
from __future__ import annotations

import re
import unicodedata
from itertools import combinations
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from nameparser import HumanName
from rapidfuzz import fuzz

from pkb.config import settings
from pkb.models.people.aggregation import IDENTIFIER_COLUMNS
from pkb.tasks.build.people.bionomia import BionomiaPeopleTask
from pkb.tasks.build.people.harvard import HarvardIndexPeopleTask
from pkb.tasks.build.people.wikidata import WikiDataPeopleTask

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

OUTPUT_DIR = Path("notebooks/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Source People

These are already built source-record tables. At this stage they are not yet reconciled into PKB Person identities.

In [ ]:
SOURCE_TASKS = {
    "wikidata": WikiDataPeopleTask(),
    "bionomia": BionomiaPeopleTask(),
    "harvard_index": HarvardIndexPeopleTask(),
}

missing = [name for name, task in SOURCE_TASKS.items() if not Path(task.output().path).exists()]
if missing:
    raise FileNotFoundError(
        "Missing built people parquet(s): "
        + ", ".join(missing)
        + ". Run: uv run luigi --module pkb.tasks.build.people.people PeopleTask --local-scheduler"
    )

source_people = pd.concat(
    [pd.read_parquet(task.output().path) for task in SOURCE_TASKS.values()],
    ignore_index=True,
    sort=False,
)

source_people = source_people.copy()
source_people["sourcePersonKey"] = source_people["pkbPersonKey"].astype(str)

print(source_people.shape)
source_people.groupby("source").size().sort_values(ascending=False)

## Name And Date Normalisation

The normalisation here is deliberately transparent. It removes accents, normalises punctuation and whitespace, handles `Surname, Given` names, keeps surname/initial signals, and extracts comparable year values.

This should be treated as methodology code until the rules are stable enough to move to `pkb.models.people.names`.

In [ ]:
TITLE_WORDS = {
    "dr", "doctor", "prof", "professor", "sir", "dame", "mr", "mrs", "ms", "miss",
}

SUFFIX_WORDS = {
    "jr", "sr", "ii", "iii", "iv", "phd", "md", "fba", "frs",
}


def clean_scalar(value):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    value = str(value).strip()
    if not value or value.lower() == "nan":
        return None
    return value


def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(c for c in text if not unicodedata.combining(c))


def canonicalise_name(value):
    value = clean_scalar(value)
    if not value:
        return None

    value = strip_accents(value)
    parsed = HumanName(value)
    if parsed.last and parsed.first:
        value = " ".join(part for part in [parsed.first, parsed.middle, parsed.last] if part)
    elif "," in value:
        surname, given = [part.strip() for part in value.split(",", 1)]
        value = f"{given} {surname}"

    value = re.sub(r"[^A-Za-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).lower().strip()
    tokens = [token for token in value.split() if token not in TITLE_WORDS | SUFFIX_WORDS]

    return " ".join(tokens) or None


def parse_name(value):
    canonical = canonicalise_name(value)
    if not canonical:
        return {
            "canonicalName": None,
            "nameTokens": [],
            "initials": None,
            "firstToken": None,
            "lastToken": None,
        }

    tokens = canonical.split()
    return {
        "canonicalName": canonical,
        "nameTokens": tokens,
        "initials": "".join(token[0] for token in tokens if token),
        "firstToken": tokens[0] if tokens else None,
        "lastToken": tokens[-1] if tokens else None,
    }


def clean_year(value):
    value = clean_scalar(value)
    if not value:
        return pd.NA
    match = re.search(r"([+-]?\d{3,4})", value)
    if not match:
        return pd.NA
    return int(match.group(1).replace("+", ""))


name_parts = source_people["name"].apply(parse_name).apply(pd.Series)
people = pd.concat([source_people, name_parts], axis=1)
for column in ["birthYear", "deathYear", "dateOfBirth", "dateOfDeath"]:
    if column not in people.columns:
        people[column] = pd.NA

people["birthYearClean"] = people["birthYear"].combine_first(people["dateOfBirth"]).apply(clean_year).astype("Int64")
people["deathYearClean"] = people["deathYear"].combine_first(people["dateOfDeath"]).apply(clean_year).astype("Int64")
people[["source", "sourcePersonKey", "name", "canonicalName", "initials", "lastToken", "birthYearClean", "deathYearClean"]].head()

## Identifier-Derived Validation Pairs

Positive pairs are source records connected by shared stable identifiers. Hard negative pairs are records that share a blocking key, such as surname, but have conflicting known identifiers.

This is a validation set for tuning. It should be manually sampled before being treated as gold standard.

In [ ]:
def has_value(value):
    if value is None:
        return False
    if isinstance(value, list):
        return any(has_value(item) for item in value)
    if isinstance(value, float) and pd.isna(value):
        return False
    value = str(value).strip()
    return bool(value) and value.lower() != "nan"


def shared_identifier_columns(a, b):
    shared = []
    for column in IDENTIFIER_COLUMNS:
        if column not in a.index or column not in b.index:
            continue
        if has_value(a[column]) and has_value(b[column]) and str(a[column]) == str(b[column]):
            shared.append(column)
    return shared


def conflicting_identifier_columns(a, b):
    conflicts = []
    for column in IDENTIFIER_COLUMNS:
        if column not in a.index or column not in b.index:
            continue
        if has_value(a[column]) and has_value(b[column]) and str(a[column]) != str(b[column]):
            conflicts.append(column)
    return conflicts


def pair_row(a, b, *, is_match, evidence):
    return {
        "sourcePersonKeyA": a.sourcePersonKey,
        "sourcePersonKeyB": b.sourcePersonKey,
        "sourceA": a.source,
        "sourceB": b.source,
        "nameA": a.name,
        "nameB": b.name,
        "canonicalNameA": a.canonicalName,
        "canonicalNameB": b.canonicalName,
        "initialsA": a.initials,
        "initialsB": b.initials,
        "lastTokenA": a.lastToken,
        "lastTokenB": b.lastToken,
        "birthYearA": a.birthYearClean,
        "birthYearB": b.birthYearClean,
        "deathYearA": a.deathYearClean,
        "deathYearB": b.deathYearClean,
        "isMatch": is_match,
        "validationEvidence": evidence,
    }


positive_rows = []
for column in IDENTIFIER_COLUMNS:
    if column not in people.columns:
        continue
    linked = people[people[column].apply(has_value)]
    for _, group in linked.groupby(column, dropna=True):
        rows = list(group.itertuples(index=False))
        for a, b in combinations(rows, 2):
            if a.sourcePersonKey != b.sourcePersonKey:
                positive_rows.append(pair_row(a, b, is_match=True, evidence=[column]))

positive_pairs = pd.DataFrame(positive_rows).drop_duplicates(["sourcePersonKeyA", "sourcePersonKeyB"])
positive_pairs.shape

In [ ]:
negative_rows = []
for _, group in people[people["lastToken"].notna()].groupby("lastToken"):
    if len(group) < 2 or len(group) > 200:
        continue
    rows = list(group.itertuples(index=False))
    for a, b in combinations(rows, 2):
        if a.sourcePersonKey == b.sourcePersonKey:
            continue
        a_s = pd.Series(a._asdict())
        b_s = pd.Series(b._asdict())
        conflicts = conflicting_identifier_columns(a_s, b_s)
        shared = shared_identifier_columns(a_s, b_s)
        if conflicts and not shared:
            negative_rows.append(pair_row(a, b, is_match=False, evidence=conflicts))

negative_pairs = pd.DataFrame(negative_rows).drop_duplicates(["sourcePersonKeyA", "sourcePersonKeyB"])
validation_pairs = pd.concat([positive_pairs, negative_pairs], ignore_index=True, sort=False)
validation_pairs.to_parquet(OUTPUT_DIR / "people_identifier_validation_pairs.parquet", index=False)

validation_pairs["isMatch"].value_counts(dropna=False)

## Rule-Based Name/Date Matching

The first production rules should be conservative. Names alone should usually produce candidates, not automatic merges.

In [ ]:
def years_equal(a, b):
    return pd.notna(a) and pd.notna(b) and int(a) == int(b)


def names_compatible(row):
    if not row.canonicalNameA or not row.canonicalNameB:
        return False
    if row.lastTokenA != row.lastTokenB:
        return False

    tokens_a = str(row.canonicalNameA).split()
    tokens_b = str(row.canonicalNameB).split()
    shorter, longer = sorted([tokens_a, tokens_b], key=len)

    for index, token in enumerate(shorter):
        if index >= len(longer):
            return False
        other = longer[index]
        if len(token) == 1:
            if token != other[:1]:
                return False
        elif token != other:
            return False
    return True


def score_person_pair(row):
    evidence = []
    name_a = row.canonicalNameA or ""
    name_b = row.canonicalNameB or ""
    similarity = max(
        fuzz.token_sort_ratio(name_a, name_b),
        fuzz.token_set_ratio(name_a, name_b),
    )
    score = 0

    if name_a and name_b and name_a == name_b:
        score += 70
        evidence.append("exact_canonical_name")
    elif similarity >= 95:
        score += 60
        evidence.append("very_high_name_similarity")
    elif similarity >= 90:
        score += 50
        evidence.append("high_name_similarity")
    elif similarity >= 85:
        score += 35
        evidence.append("moderate_name_similarity")

    if row.lastTokenA and row.lastTokenA == row.lastTokenB:
        score += 15
        evidence.append("surname_match")

    if row.initialsA and row.initialsA == row.initialsB:
        score += 15
        evidence.append("initials_match")

    if names_compatible(row):
        score += 20
        evidence.append("compatible_name_tokens")

    if years_equal(row.birthYearA, row.birthYearB):
        score += 25
        evidence.append("birth_year_match")

    if years_equal(row.deathYearA, row.deathYearB):
        score += 25
        evidence.append("death_year_match")

    return pd.Series({
        "nameSimilarity": similarity,
        "matchScore": score,
        "matchEvidence": evidence,
    })


validation_scores = validation_pairs.apply(score_person_pair, axis=1)
scored_validation = pd.concat([validation_pairs, validation_scores], axis=1)
scored_validation.head()

## Compare Rule Performance

Use the identifier-derived pairs to estimate recall against known positives and false-positive rate against hard negatives.

In [ ]:
threshold_rows = []
for threshold in [40, 50, 60, 70, 80, 90, 100, 110, 120]:
    predicted = scored_validation["matchScore"] >= threshold
    actual = scored_validation["isMatch"].astype(bool)

    true_positive = int((predicted & actual).sum())
    false_positive = int((predicted & ~actual).sum())
    false_negative = int((~predicted & actual).sum())
    true_negative = int((~predicted & ~actual).sum())

    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else np.nan
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else np.nan
    false_positive_rate = false_positive / (false_positive + true_negative) if false_positive + true_negative else np.nan

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "falsePositiveRate": false_positive_rate,
        "truePositive": true_positive,
        "falsePositive": false_positive,
        "falseNegative": false_negative,
        "trueNegative": true_negative,
    })

evaluation = pd.DataFrame(threshold_rows)
evaluation.to_parquet(OUTPUT_DIR / "people_rule_match_evaluation.parquet", index=False)
evaluation

## Generate Candidate Pairs

This blocks by surname to avoid comparing every person to every other person. Large surname blocks are skipped here for exploration; tune the cap once the data distribution is clearer.

In [ ]:
candidate_rows = []
max_block_size = 200

for last_token, group in people[people["lastToken"].notna()].groupby("lastToken"):
    if len(group) < 2 or len(group) > max_block_size:
        continue
    rows = list(group.itertuples(index=False))
    for a, b in combinations(rows, 2):
        if a.sourcePersonKey == b.sourcePersonKey:
            continue
        if a.source == b.source:
            continue
        candidate_rows.append(pair_row(a, b, is_match=None, evidence=["same_surname_block"]))

candidate_pairs = pd.DataFrame(candidate_rows).drop_duplicates(["sourcePersonKeyA", "sourcePersonKeyB"])
candidate_scores = candidate_pairs.apply(score_person_pair, axis=1)
scored_candidates = pd.concat([candidate_pairs, candidate_scores], axis=1)
scored_candidates.to_parquet(OUTPUT_DIR / "people_rule_match_candidates.parquet", index=False)

scored_candidates.sort_values("matchScore", ascending=False).head(25)

## Inspect Errors

These cells are the most important part of the notebook. Look at false positives and false negatives before promoting any rule into `PersonAggregator`.

In [ ]:
threshold = 100

false_positives = scored_validation[(scored_validation["matchScore"] >= threshold) & (~scored_validation["isMatch"].astype(bool))]
false_negatives = scored_validation[(scored_validation["matchScore"] < threshold) & (scored_validation["isMatch"].astype(bool))]

print("false positives", len(false_positives))
print("false negatives", len(false_negatives))

false_positives.sort_values("matchScore", ascending=False).head(50)

In [ ]:
false_negatives.sort_values("matchScore").head(50)

## Candidate Automatic Merge Rules

Start conservative. A possible first rule set:

- exact canonical name plus matching birth year
- exact canonical name plus matching death year
- compatible name tokens plus matching birth and death years
- very high name similarity plus matching birth and death years

Names alone should remain candidates until manually evaluated.

In [ ]:
def conservative_auto_match(row):
    exact_name = bool(row.canonicalNameA and row.canonicalNameA == row.canonicalNameB)
    birth_match = years_equal(row.birthYearA, row.birthYearB)
    death_match = years_equal(row.deathYearA, row.deathYearB)

    if exact_name and (birth_match or death_match):
        return True
    if names_compatible(row) and birth_match and death_match:
        return True
    if row.nameSimilarity >= 95 and birth_match and death_match:
        return True
    return False


scored_validation["autoMatch"] = scored_validation.apply(conservative_auto_match, axis=1)
pd.crosstab(scored_validation["isMatch"], scored_validation["autoMatch"], margins=True)